In [2]:
# Install dependencies
%pip install anthropic python-dotenv



Note: you may need to restart the kernel to use updated packages.


In [3]:
# Load env variables
from dotenv import load_dotenv
load_dotenv()

# Create an API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"



In [8]:


# Multi-turn conversation functions
def add_user_message(messages, user_prompt):
    """Add a user message to the messages list."""
    messages.append({
        "role": "user",
        "content": user_prompt
    })

def add_assistant_message(messages, assistant_response):
    """Add an assistant message to the messages list."""
    messages.append({
        "role": "assistant",
        "content": assistant_response
    })



def chat(messages, system = None, temperature = 1.0, stop_sequences = None):
    """Send a multi-turn conversation to the Anthropic client and return the assistant's response."""
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences
    }
    # You cannot pass a None into client.messages.create() for the system prompt, so only add it if it's not None.
    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [5]:
messages = []


add_user_message(messages, "Generate a very short event bridge rule as JSON")

answer = chat(messages)
# Markdown text is returned, so you can use the IPython display module to render it as Markdown in a Jupyter notebook.
answer

'```json\n{\n  "Name": "MyRule",\n  "EventBusName": "default",\n  "EventPattern": {\n    "source": ["aws.ec2"],\n    "detail-type": ["EC2 Instance State-change Notification"],\n    "detail": {\n      "state": ["running"]\n    }\n  },\n  "State": "ENABLED",\n  "Targets": [\n    {\n      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:MyFunction",\n      "Id": "1"\n    }\n  ]\n}\n```'

In [12]:
messages = []


add_user_message(messages, "Count from 1 to 10")

answer = chat(messages, stop_sequences=["5", "3, 4"])

answer

'1\n2\n3\n4\n'

In [15]:
# Markdown text is returned from event bridge, so you can use the IPython display module to render it as Markdown in a Jupyter notebook.

messages = []


add_user_message(messages, "Generate a very short event bridge rule as JSON")


answer = chat(messages)

answer

'```json\n{\n  "Name": "MyEventRule",\n  "EventBusName": "default",\n  "EventPattern": {\n    "source": ["aws.ec2"],\n    "detail-type": ["EC2 Instance State-change Notification"],\n    "detail": {\n      "state": ["running"]\n    }\n  },\n  "State": "ENABLED",\n  "Targets": [\n    {\n      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:MyFunction",\n      "Id": "1"\n    }\n  ]\n}\n```'

In [ ]:
import json

# Markdown text is returned from event bridge, so you can use the IPython display module to render it as Markdown in a Jupyter notebook.

messages = []


add_user_message(messages, "Generate a very short event bridge rule as JSON")

# Strip out the markdown with an assistant message and stop_sequences.
# First make Claude think that it already output ```json, so that it will not output it again.`
add_assistant_message(messages, "```json")
# Claude is going to think that it needs to close out the markdown block with ``` , so we can use that as a stop sequence to prevent that.
answer = chat(messages, stop_sequences=["```"])

# Now get rid of the newlines to get a valid JSON object.
json.loads(answer.strip())

{'Name': 'MyEventRule',
 'EventBusName': 'default',
 'EventPattern': {'source': ['aws.ec2'],
  'detail-type': ['EC2 Instance State-change Notification'],
  'detail': {'state': ['running']}},
 'State': 'ENABLED',
 'Targets': [{'Arn': 'arn:aws:lambda:us-east-1:123456789012:function:MyFunction',
   'Id': '1'}]}

In [ ]:


messages = []

prompt = """
Generate three different sample AWS CLI commands.  Each should be very short.
"""

add_user_message(messages, prompt)

answer = chat(messages)
answer.strip()

from IPython.display import Markdown
Markdown(answer)

# Three Sample AWS CLI Commands

1. **List all S3 buckets:**
```bash
aws s3 ls
```

2. **Describe EC2 instances:**
```bash
aws ec2 describe-instances
```

3. **Get current AWS account ID:**
```bash
aws sts get-caller-identity
```

In [ ]:
# Use message prefilling and stop sequences only to get the 3 different commands in a single response
# There shouln't be any comments or explanations in the response, just the 3 commands.
# Hint:  message prefilling isn't limited to just characters like ```

messages = []

prompt = """
Generate three different sample AWS CLI commands.  Each should be very short.
"""

add_user_message(messages, prompt)


# Let's see the raw output from Claude and then use the next cell for our work.
answer = chat(messages)
answer.strip()



'# Three Sample AWS CLI Commands\n\n1. **List all S3 buckets:**\n```bash\naws s3 ls\n```\n\n2. **Describe EC2 instances:**\n```bash\naws ec2 describe-instances\n```\n\n3. **Get current AWS account ID:**\n```bash\naws sts get-caller-identity\n```'

In [32]:
# Use message prefilling and stop sequences only to get the 3 different commands in a single response
# There shouln't be any comments or explanations in the response, just the 3 commands.
# Hint:  message prefilling isn't limited to just characters like ```
messages = []

prompt = """
Generate three different sample AWS CLI commands.  Each should be very short.
"""

add_user_message(messages, prompt)

# Get rid of the leading ```bash.  However, this will work erratically.  We'll sometimes get a single command
# and other times get bash comments.
# add_assistant_message(messages, "```bash")

# Let's use a trick to think that Claude is telling itself some instructions.
# Claude think that it's got restrictions on what it can output.
# This way you get all 3 commands and you don't get any bash command comments either.
add_assistant_message(messages, "Here are all 3 commands in a single block without an comments:\n```bash")
# 
answer = chat(messages, stop_sequences=["```"])
answer.strip()


'aws s3 ls\naws ec2 describe-instances\naws lambda list-functions'